In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Offroad_Segmentation_Training_Dataset.zip" .

In [3]:
!unzip Offroad_Segmentation_Training_Dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000123.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000124.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000125.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000126.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000127.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000128.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000129.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000130.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000131.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt10000132.png  
  inflating: Offroad_Segmentation_Training_Dataset/train/Color_Images/mt1000013

In [4]:
from google.colab import files
files.upload()


Saving train_segmentation_v2.py to train_segmentation_v2.py
Saving train_segmentation.py to train_segmentation.py
Saving test_segmentation.py to test_segmentation.py
Saving visualize.py to visualize.py


{'train_segmentation_v2.py': b'"""\r\ntrain_segmentation_v2.py\r\nImproved training \xe2\x80\x94 targets 85%+ IoU\r\nDrop-in replacement for train_segmentation.py (original files unchanged)\r\n"""\r\n\r\nimport torch\r\nfrom torch.utils.data import Dataset, DataLoader\r\nimport numpy as np\r\nfrom torch import nn\r\nimport torch.nn.functional as F\r\nimport matplotlib.pyplot as plt\r\nimport torch.optim as optim\r\nimport torchvision.transforms as transforms\r\nimport torchvision.transforms.functional as TF\r\nfrom PIL import Image\r\nimport cv2\r\nimport os\r\nimport random\r\nfrom tqdm import tqdm\r\n\r\nplt.switch_backend(\'Agg\')\r\n\r\n# \xe2\x94\x80\xe2\x94\x80 Reuse value_map exactly as original \xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80\xe2\x94\x80

In [5]:
!python train_segmentation_v2.py

Device: cuda
Traceback (most recent call last):
  File "/content/train_segmentation_v2.py", line 328, in <module>
    main()
  File "/content/train_segmentation_v2.py", line 183, in main
    trainset     = AugmentedMaskDataset(data_dir, (h, w), augment=True)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/train_segmentation_v2.py", line 47, in __init__
    self.data_ids  = os.listdir(self.image_dir)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/../Offroad_Segmentation_Training_Dataset/train/Color_Images'


In [6]:
import os
# Check where your dataset actually is
for item in os.listdir('/content'):
    print(item)

.config
visualize.py
train_stats_v2
Offroad_Segmentation_Training_Dataset.zip
train_segmentation.py
drive
train_segmentation_v2.py
test_segmentation.py
Offroad_Segmentation_Training_Dataset
sample_data


In [8]:
# Find correct path and patch the script
import re

with open('train_segmentation_v2.py', 'r') as f:
    code = f.read()

# Fix data paths to point inside /content/ directly
code = code.replace(
    "os.path.join(script_dir, '..', 'Offroad_Segmentation_Training_Dataset', 'train')",
    "os.path.join('/content', 'Offroad_Segmentation_Training_Dataset', 'train')"
)
code = code.replace(
    "os.path.join(script_dir, '..', 'Offroad_Segmentation_Training_Dataset', 'val')",
    "os.path.join('/content', 'Offroad_Segmentation_Training_Dataset', 'val')"
)

with open('train_segmentation_v2.py', 'w') as f:
    f.write(code)

print("Path fixed! Now run: !python train_segmentation_v2.py")


Path fixed! Now run: !python train_segmentation_v2.py


In [9]:
!python train_segmentation_v2.py

Device: cuda
Train: 2857  |  Val: 317

Loading DINOv2 ViT-Base backbone...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_reg4_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_reg4_pretrain.pth
100% 330M/330M [00:04<00:00, 70.4MB/s]
Embedding dim : 768
Backbone loaded successfully!

Star

In [10]:
# Run this cell ONCE to add final evaluation display to v2 script

patch = '''

    # ── Final evaluation display (matching original output format) ────────────
    import torch.nn.functional as F

    print("\\nFinal evaluation results:")

    final_losses, final_ious, final_dices, final_accs = [], [], [], []

    classifier.eval()
    backbone.eval()

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            feats   = backbone.forward_features(imgs)["x_norm_patchtokens"]
            logits  = classifier(feats)
            outputs = F.interpolate(logits, size=imgs.shape[2:],
                                    mode="bilinear", align_corners=False)

            # Loss
            final_losses.append(criterion(outputs, labels).item())

            # IoU
            final_ious.append(compute_iou(outputs, labels))

            # Dice
            pred = torch.argmax(outputs, dim=1).view(-1)
            tgt  = labels.view(-1)
            dice_scores = []
            for c in range(n_classes):
                p = (pred == c).float()
                t = (tgt  == c).float()
                inter = (p * t).sum()
                dice_scores.append(((2 * inter + 1e-6) / (p.sum() + t.sum() + 1e-6)).item())
            final_dices.append(float(np.mean(dice_scores)))

            # Accuracy
            final_accs.append((torch.argmax(outputs, dim=1) == labels).float().mean().item())

    print(f"  Final Val Loss:     {np.mean(final_losses):.4f}")
    print(f"  Final Val IoU:      {np.nanmean(final_ious):.4f}")
    print(f"  Final Val Dice:     {np.mean(final_dices):.4f}")
    print(f"  Final Val Accuracy: {np.mean(final_accs):.4f}")
'''

with open('train_segmentation_v2.py', 'r') as f:
    code = f.read()

# Insert before the last line (if __name__ == "__main__")
code = code.replace(
    "\nif __name__ == \"__main__\":",
    patch + "\nif __name__ == \"__main__\":"
)

with open('train_segmentation_v2.py', 'w') as f:
    f.write(code)

print("Done! Now run: !python train_segmentation_v2.py")

Done! Now run: !python train_segmentation_v2.py


In [11]:
!python train_segmentation_v2.py

Device: cuda
Train: 2857  |  Val: 317

Loading DINOv2 ViT-Base backbone...
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Embedding dim : 768
Backbone loaded successfully!

Starting training — max 20 epochs | early stopping patience = 5
Traceback (most recent call last):
  File "/content/train_segmentation_v2.py", line 372, in <module>
    main()
  File "/content/train_segmentation_v2.py", line 256, in main
    out

In [12]:
with open('train_segmentation_v2.py', 'r') as f:
    code = f.read()

# Remove the duplicate F import that was added by the patch
code = code.replace(
    "    # ── Final evaluation display (matching original output format) ────────────\n    import torch.nn.functional as F\n",
    "    # ── Final evaluation display (matching original output format) ────────────\n"
)

with open('train_segmentation_v2.py', 'w') as f:
    f.write(code)

print("Fixed! Now run: !python train_segmentation_v2.py")

Fixed! Now run: !python train_segmentation_v2.py


In [13]:
!python train_segmentation_v2.py

Device: cuda
Train: 2857  |  Val: 317

Loading DINOv2 ViT-Base backbone...
Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Embedding dim : 768
Backbone loaded successfully!

Starting training — max 20 epochs | early stopping patience = 5
Epoch   1/20 | Train Loss: 0.8508 | Val Loss: 0.6070 | Val IoU: 0.4409
  ✓ New best IoU: 0.4409 — model saved to /content/segmentation_head_v2_best.pth
Epoch   2/20 | Train Loss: 0.